# UI visual checks (headless Chromium + axe, live, M2M)

Session 48 (build 5, wave 5-0; owner decision D-B5-2). Runtime gate for the deployed Semantic Ground Truth app; it runs for real in session 62. Runbook: `docs/runbooks/ui_visual_checks.md`.

* **Matrix:** 11 routes (`/`, `/leaderboard`, `/assistant`, `/steward`, `/steward/feedback-queue`, `/steward/conflicts`, `/steward/feedback-branches`, `/executive`, `/admin`, `/settings`, `/does-not-exist`) x `color_scheme` light/dark x viewport 1440x900 and 390x844 = **44 cells**.
* **Auth:** the M2M SPN (same secret keys as `api_contract_tests`), `Authorization: Bearer` on every request of the browser context. The SPN is an app admin in dev, so every route renders.
* **Per cell:** wait for `[data-testid$="-root"]` (fail after 20 s); console errors and failed same-origin requests; axe WCAG 2.1 AA including `color-contrast`; exactly one `h1`; `body` background equals the token (`#F9F7F4` light, `#0B2026` dark); 15 Tab presses, each focused element must show a focus ring; full-page screenshot to `<output_volume>/<run_id>/<route>__<theme>__<viewport>.png`.
* **Figma similarity** (report only): both images at 360 px width, grayscale, 1 - mean absolute difference, against `<figma_dir>/<slug>/<slug>__populated[-mobile]__<theme>.png` when it exists.
* **Writes** one row per cell to `<catalog>.<schema>.ui_visual_results` (created if missing), prints a summary table, then **fails the task** if any cell has `status = 'fail'`.

Pure logic (matrix, rules, color and focus-ring parsing, similarity) is in `ui_visual_lib.py`, tested by `tests/test_ui_visual_lib.py`.

In [0]:
%pip install --upgrade databricks-sdk playwright axe-playwright-python pillow numpy
dbutils.library.restartPython()

In [0]:
import re
import sys
import time
from pathlib import Path

dbutils.widgets.text("app_name", "dev-semantic-ground-truth", "App name")
dbutils.widgets.text("secret_scope", "semantic_ground_truth_credentials", "Secret scope")
dbutils.widgets.text("client_id_key", "m2m_client_id_dev", "M2M client_id key")
dbutils.widgets.text("client_secret_key", "m2m_client_secret_dev", "M2M client_secret key")
dbutils.widgets.text("catalog", "hls_fde_dev", "UC catalog")
dbutils.widgets.text("schema", "dev_matthew_giglia_semantic_ground_truth", "UC schema")
dbutils.widgets.text("output_volume", "/Volumes/hls_fde_dev/dev_matthew_giglia_semantic_ground_truth/ui_visual_checks", "Screenshot volume path")
dbutils.widgets.text("figma_dir", "/Workspace/Users/matthew.giglia@databricks.com/semanticGroundTruth/docs/design/ui/figma", "Figma export dir (empty = skip similarity)")
dbutils.widgets.text("run_id", "", "Job run id (empty = manual run)")

APP_NAME = dbutils.widgets.get("app_name").strip()
SECRET_SCOPE = dbutils.widgets.get("secret_scope").strip()
CLIENT_ID_KEY = dbutils.widgets.get("client_id_key").strip()
CLIENT_SECRET_KEY = dbutils.widgets.get("client_secret_key").strip()
CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
OUTPUT_VOLUME = dbutils.widgets.get("output_volume").strip().rstrip("/")
FIGMA_DIR = dbutils.widgets.get("figma_dir").strip()
RUN_ID = dbutils.widgets.get("run_id").strip() or f"manual-{int(time.time())}"

# Identifiers are interpolated into SQL below: allow only plain UC names.
for _label, _ident in (("catalog", CATALOG), ("schema", SCHEMA)):
    if not re.fullmatch(r"[A-Za-z0-9_]+", _ident):
        raise ValueError(f"invalid {_label} name: {_ident!r}")
if not APP_NAME:
    raise ValueError("app_name is required")

# Pure helpers live next to this notebook (<app root>/src/notebooks/ui_visual_lib.py).
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_nb_dir = Path(_nb_path if _nb_path.startswith("/Workspace") else "/Workspace" + _nb_path).parent
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))
import ui_visual_lib as lib  # noqa: E402

lib.parse_volume_path(OUTPUT_VOLUME)  # validates the /Volumes/<catalog>/<schema>/<volume> shape
RESULTS_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`ui_visual_results`"
CELLS = lib.build_matrix()
if FIGMA_DIR and not Path(FIGMA_DIR).is_dir():
    print(f"figma_dir {FIGMA_DIR} not found: figma_similarity will be null")
    FIGMA_DIR = ""
print(f"run {RUN_ID}: {len(CELLS)} cells against app {APP_NAME}; results -> {RESULTS_TABLE}; screenshots -> {OUTPUT_VOLUME}/{RUN_ID}/")

In [0]:
import os
import platform
import subprocess

# Serverless (Ubuntu 24.04, no root): Playwright's headless shell needs 4 libraries that are not
# installed. Download the .debs with a private apt state dir and unpack them under /tmp (no dpkg -i).
CDEPS = "/tmp/cdeps"
# Serverless hosts can be x86_64 or arm64 (session 62: run 1099658488939115 got an arm64 host and
# Chromium could not find libatk-1.0.so.0 under x86_64-linux-gnu). apt downloads the host's arch.
CDEPS_LIB = f"{CDEPS}/root/usr/lib/{'aarch64' if platform.machine() in ('aarch64', 'arm64') else 'x86_64'}-linux-gnu"
APT_PKGS = "libatk1.0-0t64 libatk-bridge2.0-0t64 libatspi2.0-0t64 libxdamage1"
RECIPE = f"""
set -euo pipefail
A=/tmp/apt; mkdir -p $A/lists/partial $A/cache/archives/partial {CDEPS}/debs {CDEPS}/root
O="-o Dir::State::Lists=$A/lists -o Dir::Cache=$A/cache -o Debug::NoLocking=1 -o APT::Sandbox::User=$(id -un)"
apt-get $O -qq update
cd {CDEPS}/debs && apt-get $O -qq download {APT_PKGS}
for d in {CDEPS}/debs/*.deb; do dpkg -x "$d" {CDEPS}/root; done
"""


def sh(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"{cmd if isinstance(cmd, str) else ' '.join(cmd)} failed ({r.returncode}):\n{r.stdout[-2000:]}\n{r.stderr[-2000:]}")
    return r.stdout


sh([sys.executable, "-m", "playwright", "install", "chromium"])
if not Path(CDEPS_LIB, "libatk-1.0.so.0").exists():
    sh(["bash", "-c", RECIPE])
# The Playwright driver (started below) inherits this environment and passes it to Chromium.
os.environ["LD_LIBRARY_PATH"] = CDEPS_LIB + (":" + os.environ["LD_LIBRARY_PATH"] if os.environ.get("LD_LIBRARY_PATH") else "")
print("chromium ready; LD_LIBRARY_PATH =", os.environ["LD_LIBRARY_PATH"])

In [0]:
import requests
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
_url = w.apps.get(APP_NAME).url or ""
APP_URL = (_url if _url.startswith("https://") else f"https://{_url}").rstrip("/")

# M2M OAuth (client credentials) as the app's admin SPN; a plain PAT gets 302/403 from the Apps proxy.
m2m_w = WorkspaceClient(
    host=w.config.host,
    client_id=dbutils.secrets.get(scope=SECRET_SCOPE, key=CLIENT_ID_KEY),
    client_secret=dbutils.secrets.get(scope=SECRET_SCOPE, key=CLIENT_SECRET_KEY),
)

# Fail fast on auth before launching 44 browser contexts.
_me = requests.get(APP_URL + "/api/me", headers=m2m_w.config.authenticate(), timeout=60, allow_redirects=False)
if _me.status_code != 200:
    raise RuntimeError(f"GET /api/me as the M2M SPN returned {_me.status_code} (302/401/403 = SPN lacks CAN_USE on the app or bad secret)")
print("M2M identity:", _me.json().get("userId"), "roles", _me.json().get("roles"))

# color_scheme emulation only changes the theme when the SPN's theme setting is `system`.
_settings = requests.get(APP_URL + "/api/me/settings", headers=m2m_w.config.authenticate(), timeout=60, allow_redirects=False)
_theme = _settings.json().get("theme") if _settings.status_code == 200 else None
if _theme != "system":
    print(f"WARNING: the M2M SPN theme setting is {_theme!r}, not 'system'; the other color scheme will fail bg_ok.")

SHOT_DIR = Path(OUTPUT_VOLUME) / RUN_ID
SHOT_DIR.mkdir(parents=True, exist_ok=True)
print(f"app {APP_URL}; screenshots in {SHOT_DIR}")

In [0]:
import io
from datetime import datetime, timezone

import axe_playwright_python
from PIL import Image
from playwright.sync_api import TimeoutError as PlaywrightTimeout
from playwright.sync_api import sync_playwright

# axe-core ships inside the axe-playwright-python wheel; inject it ourselves to control the rule set.
AXE_JS = (Path(axe_playwright_python.__file__).parent / "axe.min.js").read_text()
AXE_RUN_JS = """async (tags) => {
  const r = await axe.run(document, {runOnly: {type: 'tag', values: tags},
                                     rules: {'color-contrast': {enabled: true}}, resultTypes: ['violations']});
  return {violations: r.violations.map(v => ({id: v.id, impact: v.impact, help: v.help,
          nodes: v.nodes.slice(0, 5).map(n => ({target: n.target, summary: (n.failureSummary || '').slice(0, 300)}))}))};
}"""
FOCUS_JS = """() => {
  const el = document.activeElement;
  if (!el || el === document.body || el === document.documentElement) return {is_body: true};
  const cs = getComputedStyle(el);
  return {is_body: false, tag: el.tagName.toLowerCase(),
          label: (el.getAttribute('aria-label') || el.getAttribute('data-testid') || el.textContent || '').trim().slice(0, 60),
          style: {outlineStyle: cs.outlineStyle, outlineWidth: cs.outlineWidth, outlineColor: cs.outlineColor, boxShadow: cs.boxShadow}};
}"""

ROWS, DETAILS = [], {}


def run_cell(browser, cell):
    t0 = time.monotonic()
    console, failed = [], []
    ctx = browser.new_context(
        viewport={"width": cell.width, "height": cell.height},
        color_scheme=cell.theme,
        extra_http_headers=m2m_w.config.authenticate(),  # fresh token per cell
        bypass_csp=True,  # lets us inject axe into a page with a strict CSP
    )
    page = ctx.new_page()
    page.on("console", lambda m: console.append(m.text[:300]) if m.type == "error" else None)
    page.on("pageerror", lambda e: console.append(f"pageerror: {str(e)[:300]}"))
    page.on("response", lambda r: failed.append(f"{r.status} {r.url}") if lib.is_failed_request(r.url, APP_URL, r.status) else None)
    page.on("requestfailed", lambda r: failed.append(f"{r.failure} {r.url}") if lib.is_failed_request(r.url, APP_URL, None, r.failure) else None)
    detail = {}
    rendered, h1_count, bg_ok, focus_ok, sim = False, 0, False, False, None
    axe_total = axe_serious = 0
    shot = lib.screenshot_path(OUTPUT_VOLUME, RUN_ID, cell)
    try:
        page.goto(APP_URL + cell.route, wait_until="domcontentloaded", timeout=60_000)
        try:
            page.wait_for_selector(lib.ROOT_SELECTOR, state="visible", timeout=lib.RENDER_TIMEOUT_MS)
            rendered = True
        except PlaywrightTimeout:
            pass
        try:
            page.wait_for_load_state("networkidle", timeout=10_000)
        except PlaywrightTimeout:
            pass  # long-polling or streaming; the checks below still run
        h1_count = page.locator("h1").count()
        bg = page.evaluate("() => getComputedStyle(document.body).backgroundColor")
        bg_ok = lib.bg_matches(cell.theme, bg)
        detail["bg"] = bg
        page.add_script_tag(content=AXE_JS)
        axe_total, axe_serious, detail["axe"] = lib.axe_counts(page.evaluate(AXE_RUN_JS, list(lib.AXE_TAGS)))
        png = page.screenshot(full_page=True)
        Path(shot).write_bytes(png)
        ref = lib.figma_reference_path(FIGMA_DIR, cell)
        if ref:
            sim = lib.image_similarity(Image.open(io.BytesIO(png)), Image.open(ref))
        samples = []
        page.mouse.click(1, 1)  # start the Tab order from the document, not the address bar
        for _ in range(lib.FOCUS_TABS):
            page.keyboard.press("Tab")
            page.wait_for_timeout(150)  # let focus-ring transitions finish
            samples.append(page.evaluate(FOCUS_JS))
        focus_ok, detail["focus_offenders"] = lib.focus_ok(samples)
        detail["focus_stops"] = sum(1 for s in samples if not s.get("is_body"))
    except Exception as e:  # noqa: BLE001  a broken cell is recorded as fail, the matrix continues
        detail["error"] = f"{type(e).__name__}: {str(e)[:500]}"
    finally:
        ctx.close()
    status, reasons = lib.evaluate_cell(rendered=rendered, axe_serious=axe_serious, console_errors=len(console),
                                        failed_requests=len(failed), h1_count=h1_count, bg_ok=bg_ok, focus_ok=focus_ok)
    if "error" in detail:
        status, reasons = "fail", [detail["error"]] + reasons
    detail.update(reasons=reasons, console=console[:10], failed_requests=failed[:10])
    DETAILS[(cell.route, cell.theme, cell.viewport)] = detail
    ROWS.append({
        "run_id": RUN_ID, "route": cell.route, "theme": cell.theme, "viewport": cell.viewport, "status": status,
        "axe_violations": int(axe_total), "axe_serious": int(axe_serious), "console_errors": len(console),
        "failed_requests": len(failed), "h1_count": int(h1_count), "bg_ok": bool(bg_ok), "focus_ok": bool(focus_ok),
        "figma_similarity": sim, "screenshot_path": shot if Path(shot).exists() else None,
        "duration_ms": int((time.monotonic() - t0) * 1000), "ts": datetime.now(timezone.utc),
    })
    print(f"[{status.upper()}] {cell.route} {cell.theme} {cell.viewport}" + ("" if status == "pass" else f": {'; '.join(reasons)}"))


def run_matrix():
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        try:
            for cell in CELLS:
                run_cell(browser, cell)
        finally:
            browser.close()


# The notebook kernel runs an asyncio loop, and the Playwright sync API refuses to start inside one
# ("using Playwright Sync API inside the asyncio loop"). A worker thread has no loop.
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=1) as _pool:
    _pool.submit(run_matrix).result()
print(f"{len(ROWS)} cells measured")

In [0]:
import json

DDL = ", ".join(f"{c} {t}" for c, t in lib.RESULT_COLUMNS)
spark.sql(f"CREATE TABLE IF NOT EXISTS {RESULTS_TABLE} ({DDL}) "
          "COMMENT 'ui_visual_checks job: one row per route x theme x viewport cell per run (session 48)'")
spark.createDataFrame(ROWS, schema=DDL).write.mode("append").saveAsTable(RESULTS_TABLE.replace("`", ""))

print("\n".join(lib.summary_lines(ROWS)))
failing = [r for r in ROWS if r["status"] == "fail"]
for r in failing:
    d = DETAILS[(r["route"], r["theme"], r["viewport"])]
    print(f"\n{r['route']} {r['theme']} {r['viewport']} -> {r['screenshot_path']}\n" + json.dumps(d, indent=1, default=str)[:3000])

sims = sorted((r for r in ROWS if r["figma_similarity"] is not None), key=lambda r: r["figma_similarity"])
if sims:
    print("\nlowest figma_similarity (report only):")
    for r in sims[:5]:
        print(f"  {r['figma_similarity']:.3f} {r['route']} {r['theme']} {r['viewport']} {r['screenshot_path']}")

print(f"\n{len(ROWS)} rows written to {RESULTS_TABLE} for run {RUN_ID}")
if len(ROWS) != len(CELLS):
    raise AssertionError(f"only {len(ROWS)}/{len(CELLS)} cells were measured")
if failing:
    raise AssertionError(f"{len(failing)}/{len(ROWS)} UI visual cells failed: "
                         + ", ".join(f"{r['route']} {r['theme']} {r['viewport']}" for r in failing))